# fact_ohlc_hourly

Hourly candles from Kraken (batch). `ohlc_prices` is SCD2, so only the current version of each candle (`is_current = true`) goes to gold.

Grain: **one asset x one hour**. Source: `ohlc_prices`.

Quality gate before writing: row count = distinct business keys in silver, every dimension key found,
unique primary key, every date inside `dim_date`. Any problem stops the notebook, nothing is written.

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("silver_schema", "team_crypto_silver")
dbutils.widgets.text("gold_schema", "team_crypto_gold")

CATALOG = dbutils.widgets.get("catalog")
SILVER = f"{CATALOG}.{dbutils.widgets.get('silver_schema')}"
GOLD = f"{CATALOG}.{dbutils.widgets.get('gold_schema')}"

# same time zone everywhere, so date_key and hour_of_day don't shift
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
from pyspark.sql import functions as F
from utilities.gold_transformations import FACT_OHLC_COLUMNS, build_fact_ohlc_hourly, current_ohlc, fact_quality_issues

silver_df = current_ohlc(spark.table(f"{SILVER}.ohlc_prices"))
dim_asset = spark.table(f"{GOLD}.dim_asset")
dim_source = spark.table(f"{GOLD}.dim_source")
dim_date = spark.table(f"{GOLD}.dim_date")

fact_df = build_fact_ohlc_hourly(silver_df, dim_asset, dim_source)

### Quality gate

In [0]:
expected_rows = silver_df.select("symbol", "event_time").distinct().count()

issues = fact_quality_issues(
    fact_df,
    expected_rows=expected_rows,
    key_columns=["asset_key", "event_time"],
    dim_columns=["asset_key", "date_key", "source_key"],
    calendar=dim_date,
)
if issues:
    raise ValueError("fact_ohlc_hourly quality gate failed: " + "; ".join(issues))

### Write (idempotent full overwrite)

In [0]:
(fact_df
    .select(*FACT_OHLC_COLUMNS)
    .withColumn("processed_at", F.current_timestamp())
    .createOrReplaceTempView("fact_src"))

cols = ", ".join(FACT_OHLC_COLUMNS + ["processed_at"])
spark.sql(f"INSERT OVERWRITE {GOLD}.fact_ohlc_hourly ({cols}) SELECT {cols} FROM fact_src")